In [165]:
!pip install tenacity --upgrade
!pip install pandas

In [153]:
import requests
from typing import List , Dict
import re
import pandas as pd
from tenacity import retry , stop_after_attempt , wait_exponential_jitter , RetryError

### CORE-API
core api provides access to the largest collection of thousands open access research papers from different providers

Get Apikey from `https://core.ac.uk/services/api`

In [162]:
@retry(stop=stop_after_attempt(10),wait=wait_exponential_jitter(5 , 15))  #retry if we hit ratelimit
def _send_request(endpoint: str ,topic:str , num_results:int ,api_key: str|None = None):
  res  =  requests.post(endpoint ,
                        headers={"Authorization": f"Bearer {api_key}"} if api_key else None,
                        json={"q": topic ,  "limit": num_results})
  if res.status_code == 200:
    return res.json()
  else:
    raise RuntimeError(f"couldnt access data for {topic}, finished with status code : {res.status_code}")

def send_request(endpoint: str ,topic:str , num_results:int ,api_key: str|None = None):
  try:
    result = _send_request(endpoint,topic,num_results,api_key)
    return result
  except RetryError as e:
      raise(e)

def process_data(entry: Dict) -> List:   #process and pick the field with better contextual information
  def _pick_best_field(child_entry:Dict)-> str:
      if not child_entry.get("abstract"):
        return  child_entry.get("fullText")[:2000]  #pick first 2000 words if 'abstract' field is empty
      url_pattern = r"^(https?:\/\/)?([\w\-]+\.)+[\w\-]+(\/[\w\-._~:/?#[\]@!$&'()*+,;=]*)?$"
      is_url = bool(re.match(url_pattern,child_entry.get("abstract")))
      if is_url:
        return  child_entry.get("fullText")[:2000]   #pick first 2000 words if 'abstract' field contains url
      else:
        return child_entry.get("abstract")   #pick 'abstract' only if its present and its not a url
  out =  list(map(_pick_best_field , entry.get("results"))) #run with map for perform transformation in parallel
  return out


def get_data(topics: List , num_results: int ,
             output_dir:str|None = None ,api_key : str|None = None)-> Dict[str,List]:
  """
  topics : list of topics to search for
  num_results: number of results to return
  output_dir: useful if building a dataset, to store result to dataset
  api_key: api key from -> https://core.ac.uk/services/api
  """
  endpoint = "https://api.core.ac.uk/v3/search/works"
  results =  {}
  for topic in topics:
    results[topic] = process_data(send_request(endpoint,topic,num_results,api_key))
  if output_dir:
     if output_dir.endswith(".csv"):
        pd.DataFrame.from_dict(results).to_csv(output_dir)
     else:
        raise RuntimeError("`output_dir` isnt a .csv path")

  return results


In [166]:
#usage

data = get_data(["psychology"] , 10)
data

{'psychology': ['Experiments take place in a physical environment but also a social environment. Generalizability from experimental manipulations to more typical contexts may be limited by violations of ecological validity with respect to either the physical or the social environment. A replication and extension of a recent study (a blood glucose manipulation) was conducted to investigate the effects of experimental demand (a social artifact) on participant behaviors judging the geographical slant of a large-scale outdoor hill. Three different assessments of experimental demand indicate that even when the physical environment is naturalistic, and the goal of the main experimental manipulation was primarily concealed, artificial aspects of the social environment (such as an explicit requirement to wear a heavy backpack while estimating the slant of a hill) may still be primarily responsible for altered judgments of hill orientation. (PsycINFO Database Record (c) 2013 APA, all rights res